## 2 序列模型

### 2.2 编程题：文本预处理与滑动窗口采样

文本预处理的首要任务是将输入字符串统一转换为小写形式，并利用正则表达式过滤掉所有非字母字符（仅保留字母与空格），以确保后续分词操作的准确性。词汇表的构建依赖于词频统计信息，采用频率降序排列作为主要排序依据，在频率相同时辅以字母升序，从而为每个单词分配从零开始的唯一整数索引，确保映射关系的确定性与可复现性。滑动窗口采样逻辑严格遵循自回归语言模型的数据组织需求，设定窗口长度为 n，顺序遍历分词列表，将当前窗口中所有词语组成特征序列，并将窗口紧邻的下一个词语作为预测标签。遍历过程自动终止于列表末尾，有效忽略无法构成完整窗口的残余数据，最终返回词汇表字典以及对应的特征与标签列表。

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写并去标点（仅保留字母和空格）
    text = re.sub(r'[^a-z\s]', '', text.lower())
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 按频率降序构建词汇表，ID从0开始
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.keys(), key=lambda w: (-word_counts[w], w))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征与标签
    features = []
    labels = []
    for i in range(len(words) - n):
        features.append(words[i:i+n])
        labels.append(words[i+n])
        
    return vocab, (features, labels)

# 测试用例
if __name__ == "__main__":
    vocab, (feat, lab) = preprocess_text("The time machine", 2)
    print("词汇表:", vocab)   
    print("特征:", feat)     
    print("标签:", lab)      

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']


### 3.2 编程题：单步RNN单元的正反向传播实现
在前向传播阶段，当前时刻的输入向量与上一时刻的隐藏状态分别与对应的权重矩阵相乘并加上偏置项，随后通过双曲正切激活函数进行非线性变换，从而得到当前时刻的隐藏状态。反向传播的实现则依赖于对激活函数导数的精确计算，利用 tanh 导数等于 1 减去当前输出平方的特性，将上游传递而来的梯度映射至仿射变换前的误差信号。在此基础上，依据矩阵乘法的链式法则，依次推导出对输入、上一时刻状态、输入权重、循环权重以及偏置的梯度表达式。特别需要注意的是，偏置项的梯度需要在批次维度上进行累加求和，以确保所有梯度张量的形状与前向传播中各变量的维度严格匹配。

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_xh, W_hh, b_h):
    a_t = np.dot(x_t, W_xh) + np.dot(h_prev, W_hh) + b_h
    h_t = np.tanh(a_t)
    cache = (x_t, h_prev, W_xh, W_hh, b_h, a_t, h_t)
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    x_t, h_prev, W_xh, W_hh, b_h, a_t, h_t = cache
    # tanh 的反向导数
    d_a = dh_next * (1 - h_t ** 2)
    # 各参数梯度计算
    dx_t = np.dot(d_a, W_xh.T)
    dh_prev = np.dot(d_a, W_hh.T)
    dW_xh = np.dot(x_t.T, d_a)
    dW_hh = np.dot(h_prev.T, d_a)
    db_h = np.sum(d_a, axis=0)
    return dx_t, dh_prev, dW_xh, dW_hh, db_h

# 形状验证
if __name__ == "__main__":
    batch, in_dim, hid_dim = 4, 3, 5
    x = np.random.randn(batch, in_dim)
    h = np.random.randn(batch, hid_dim)
    Wx = np.random.randn(in_dim, hid_dim)
    Wh = np.random.randn(hid_dim, hid_dim)
    b = np.random.randn(hid_dim)
    ht, cache = rnn_cell_forward(x, h, Wx, Wh, b)
    dht = np.random.randn(batch, hid_dim)
    dx, dh, dWx, dWh, db = rnn_cell_backward(dht, cache)
    print("梯度形状验证通过:", dx.shape, dWx.shape, dWh.shape, db.shape)

梯度形状验证通过: (4, 3) (3, 5) (5, 5) (5,)


### 4.2 编程题：双向RNN编码器的输出拼接实现
本实现直接调用 PyTorch 框架中的 torch.nn.RNN 模块，并通过启用双向选项来同时捕获序列中的前向与后向依赖关系。为了保证输入输出格式的规范性，将批次维度置于序列长度与特征维度之间，并设置批次优先标志为假。网络执行前向传播后，返回的输出张量在每个时间步上都包含了前向与后向隐藏状态的拼接信息，因此其特征维度自动扩展为两倍的隐藏单元数。为了获取整个序列的紧凑向量表示，直接取输出张量在序列维度上的最后一个元素，此时该时间步已经融合了全部前向信息以及完整的后向上下文，能够有效作为输入序列的全局特征表达。

In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,
            bidirectional=True
        )
    
    def forward(self, X):
        # X: (seq_len, batch, input_dim)
        output, _ = self.rnn(X)  # output: (seq_len, batch, 2*hidden_dim)
        final_state = output[-1, :, :]  # (batch, 2*hidden_dim)
        return output, final_state

# 测试用例
if __name__ == "__main__":
    seq, batch, inp = 5, 3, 4
    hidden = 6
    X = torch.randn(seq, batch, inp)
    encoder = BidirectionalRNNEncoder(inp, hidden)
    out, final = encoder(X)
    print("每个时间步输出形状:", out.shape)    
    print("最终序列表示形状:", final.shape)    

每个时间步输出形状: torch.Size([5, 3, 12])
最终序列表示形状: torch.Size([3, 12])


### 5.2 编程题：CBOW完整Softmax损失计算
对于给定批次中的每个训练样本，首先依据上下文词的索引从输入嵌入矩阵中逐一取出对应的词向量，并在上下文数量维度上进行算术平均操作，从而获得压缩后的隐藏层向量表示。随后，将该隐藏向量与输出权重矩阵进行矩阵乘法运算，得到词汇表大小的原始分数向量。为了防止指数运算引发数值溢出，在实施 Softmax 归一化之前需要对分数向量执行最大值平移操作，从而获得稳定的概率分布。最后，根据目标中心词的索引取出对应的概率值并计算其负对数，作为当前样本的交叉熵损失，并将批次内所有样本的损失值取平均后返回。

In [4]:
import numpy as np

def cbow_forward(context_indices, target_idx, W, W_out):
    """
    context_indices: list of lists, 每个子列表长度为 context_size
    target_idx: list of ints, 目标词索引
    W: (V, d) 输入嵌入
    W_out: (d, V) 输出权重
    """
    batch_size = len(context_indices)
    total_loss = 0.0
    
    for i in range(batch_size):
        # 平均上下文向量
        ctx_vecs = W[context_indices[i], :]  # (context_size, d)
        h = np.mean(ctx_vecs, axis=0)        # (d,)
        # 计算 logits 并应用稳定 Softmax
        logits = np.dot(h, W_out)            # (V,)
        max_log = np.max(logits)
        probs = np.exp(logits - max_log) / np.sum(np.exp(logits - max_log))
        # 累积交叉熵
        total_loss += -np.log(probs[target_idx[i]] + 1e-10)
        
    return total_loss / batch_size

# 测试用例
if __name__ == "__main__":
    V, d = 10, 5
    context_size = 3
    batch = 4
    W_emb = np.random.randn(V, d)
    W_out = np.random.randn(d, V)
    ctx_samples = [np.random.randint(0, V, size=context_size).tolist() for _ in range(batch)]
    targets = np.random.randint(0, V, size=batch).tolist()
    loss_val = cbow_forward(ctx_samples, targets, W_emb, W_out)
    print("CBOW 平均损失值:", loss_val)

CBOW 平均损失值: 2.3460088724368573


### 6.2 编程题：多头注意力的并行化前向传播
输入序列首先经过三组独立的线性投影变换，分别生成查询、键和值张量，其维度均为序列长度乘以批次大小乘以模型维度。接着通过维度变形与转置操作，将张量拆解为多个独立的注意力头，每个头负责处理原始特征空间中的一个子空间，从而实现多头并行计算。在每个头内部，缩放点积注意力机制并行执行，依次完成得分计算、Softmax 归一化以及加权聚合操作。完成所有头的注意力计算后，将各头的输出沿特征维度拼接，并通过最终的输出投影层进行线性变换，从而将多头信息融合为统一的表示，最终输出张量在形状上与原始输入完全一致。

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, X):
        seq_len, batch, _ = X.shape
        # 1. 投影
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        # 2. 变形为多头的形式 (batch, heads, seq, d_k)
        Q = Q.transpose(0, 1).view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.transpose(0, 1).view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.transpose(0, 1).view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        # 3. 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_weights, V)
        # 4. 拼接恢复
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        attn_out = attn_out.transpose(0, 1)  # 回到 (seq, batch, d_model)
        # 5. 最终线性层
        output = self.W_o(attn_out)
        return output

# 测试用例
if __name__ == "__main__":
    seq, batch, d_model = 5, 3, 4
    heads = 2
    X = torch.randn(seq, batch, d_model)
    mha = MultiHeadAttention(d_model, heads)
    out = mha(X)
    print("多头注意力输出形状:", out.shape) 

多头注意力输出形状: torch.Size([5, 3, 4])
